<a href="https://colab.research.google.com/github/hig3r/mva/blob/2026/E06/mva_2026_E06_03_GMM_generate_visualize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mva_2026_E06_03_GMM_generate_visualize 混合ガウス分布の生成・可視化

In [ ]:
import numpy as np # 数値計算ライブラリ
import pandas as pd # データ解析ライブラリ
import matplotlib.pyplot as plt # グラフ描画ライブラリ
%matplotlib inline
import plotly.graph_objects as go # 3次元インタラクティブグラフィクス
from mpl_toolkits.mplot3d import Axes3D

from scipy import stats
from sklearn.mixture import GaussianMixture

%pip install japanize_matplotlib
import japanize_matplotlib # 日本語フォント

In [ ]:
np.random.seed(42)

## 1次元データサンプル生成

In [ ]:
# サンプルサイズ
n0 = 300
n1 = 200
n = n0+n1

In [ ]:
# 1次元データ
mu0=-1
mu1=2
sigma0=0.8
sigma1=1.2

### 方法1: scipy.stats.norm を2回使う

In [ ]:
rvx0=stats.norm(loc=mu0, scale=sigma0)
rvx1=stats.norm(loc=mu1, scale=sigma1)
X0 = rvx0.rvs(size=n0)
X1 = rvx1.rvs(size=n1)

# データを結合する
X = np.concatenate([X0, X1])

# 最小値と最大値から共通のビン（区切り）を定義する
bins = np.linspace(X.min(), X.max(), 30)

# データの分布を可視化 (さらにalphaを下げて透明度を調整)
plt.figure(figsize=(8, 5))
plt.hist(X0, bins=bins, density=False, alpha=0.20, color='r', edgecolor='black', label='X0 (Class 0)')
plt.hist(X1, bins=bins, density=False, alpha=0.20, color='b', edgecolor='black', label='X1 (Class 1)')
plt.hist(X, bins=bins, density=False, alpha=0.08, color='g', edgecolor='black', label='X (混合全体)')
plt.title('1次元混合ガウス分布')
plt.xlabel('x')
plt.ylabel('頻度')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
print(X0[0:5])
print(X1[0:5])
print(X[0:5])

### 方法2: sklearn.mixture.GaussianMixtureを使う

In [ ]:
gmm = GaussianMixture(n_components=2) #0,1の2個

# pi_0, pi_1
gmm.weights_ = np.array([300/(300+200), 200/(300+200)])

gmm.means_ = np.array([[mu0],
                       [mu1]])

gmm.covariances_ = np.array([[[sigma0*sigma0]],
                             [[sigma1*sigma1]]])

X_sampled, y_sampled = gmm.sample(n)


bins = np.linspace(X.min(), X.max(), 30)
# データの確認と可視化
plt.figure(figsize=(8, 5))
plt.hist(X_sampled, bins=bins, density=False, alpha=0.6, color='skyblue', edgecolor='black', label='x')

# 各コンポーネント（どのガウス分布から生成されたか）ごとに色分けしてプロット
plt.scatter(X_sampled, np.full_like(X_sampled, -0.02), c=y_sampled, cmap='coolwarm', alpha=0.5, label='(x,y)')

plt.title('1次元混合ガウス分布')
plt.xlabel('x')
plt.ylabel('頻度')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
print(X_sampled[:5])
print(y_sampled[:5])

### 確率密度関数（母分布）

In [ ]:
# 1次元用の可視化データ範囲を定義
x_axis = np.linspace(X.min() - 1, X.max() + 1, 1000)

# 各コンポーネント（母分布）のPDFを計算
pdf0 = rvx0.pdf(x_axis)
pdf1 = rvx1.pdf(x_axis)

# 混合比率（重み）に基づき全体のPDFを計算
pi0 = n0 / n
pi1 = n1 / n
pdf_mixture = pi0 * pdf0 + pi1 * pdf1

# 描画
plt.figure(figsize=(10, 6))
plt.plot(x_axis, pdf0, 'r--', label='Class 0 PDF ($x_0$)', alpha=0.8)
plt.plot(x_axis, pdf1, 'b--', label='Class 1 PDF ($x_1$)', alpha=0.8)
plt.plot(x_axis, pdf_mixture, 'g-', linewidth=2, label=r'混合母分布 ($\pi_0 x_0 + \pi_1 x_1$)', alpha=0.9)

# データのヒストグラムも薄く重ねる
plt.hist(X0, bins=bins, density=True, alpha=0.1, color='r', edgecolor='black')
plt.hist(X1, bins=bins, density=True, alpha=0.1, color='b', edgecolor='black')

plt.scatter(X_sampled, np.full_like(X_sampled, -0.02), c=y_sampled, cmap='coolwarm', alpha=0.5, label='(x,y)')

plt.title('1次元混合ガウス分布の確率密度関数（母分布）')
plt.xlabel('x')
plt.ylabel('確率密度')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig("gmm1d-discr.pdf", format='pdf', bbox_inches='tight')
plt.show()

## 2次元データ

In [ ]:
# サンプルサイズ
n0 = 300
n1 = 200
n = n0+n1

In [ ]:
pi= np.array([0.6,0.4])
mean=np.array([[-1.0, -1.0],
                [0.5, 1]])
cov0 = np.array([[0.8, 0.3],
                  [0.3, 0.6]])
cov1 = np.array([[1.2, -0.4],
                  [-0.4, 0.9]])
cov=[cov0,cov1]

### 方法1: scipy.stats.multivariate_normal を2回使う

In [ ]:
rvx20=stats.multivariate_normal(mean=mean[0], cov=cov[0])
rvx21=stats.multivariate_normal(mean=mean[1], cov=cov[1])
# 前回の修正で rvx0 / rvx1 を参照していたバグを修正し、作成した 2D の分布からサンプリングします
X20 = rvx20.rvs(size=n0)
X21 = rvx21.rvs(size=n1)

# データを結合する
X2 = np.concatenate([X20, X21])

In [ ]:
print(X20[:5])
print(X21[:5])

In [ ]:
# 2次元データの散布図による可視化
plt.figure(figsize=(10, 8))
plt.scatter(X20[:, 0], X20[:, 1], alpha=0.4, color='r', edgecolor='black', label='X20 (Class 0)')
plt.scatter(X21[:, 0], X21[:, 1], alpha=0.4, color='b', edgecolor='black', label='X21 (Class 1)')

# 等高線を描画するためのグリッド作成
x1_min, x1_max = X2[:, 0].min() - 1, X2[:, 0].max() + 1
x2_min, x2_max = X2[:, 1].min() - 1, X2[:, 1].max() + 1
x1_grid, x2_grid = np.meshgrid(np.linspace(x1_min, x1_max, 200), np.linspace(x2_min, x2_max, 200))
pos = np.dstack((x1_grid, x2_grid))

plt.title('2次元混合ガウス分布のサンプルデータと真の確率密度等高線')
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.axis('equal')
plt.show()

### 方法2: sklearn.mixture.GaussianMixtureを使う

In [ ]:
gmm2 = GaussianMixture(n_components=2) # y=0,1 の2個

gmm2.weights_ = pi

# means_ (各正規分布の2次元平均ベクトル、形状は (n_components, n_features) = (2, 2))
gmm2.means_ = mean
gmm2.covariances_ = np.array([cov[0], cov[1]])

# 3. sample() メソッドを使って2次元データを生成
n2 = n
X2, y2 = gmm2.sample(n)

In [ ]:
print(X2[1:5])
print(y2[1:5])

In [ ]:
# データの確認と可視化（散布図）
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X2[:, 0], X2[:, 1], c=y2, cmap='coolwarm', alpha=0.7, edgecolors='k')
plt.title('2次元混合ガウス分布')
plt.xlabel('x1')
plt.ylabel('x2')
plt.colorbar(scatter, label='y')
plt.grid(True, linestyle='--', alpha=0.5)
plt.axis('equal')
plt.show()

### 確率密度関数（母分布）

In [ ]:
# 2次元データの散布図による可視化
plt.figure(figsize=(10, 8))
# 各成分の確率密度（PDF）の計算
pdf0 = rvx20.pdf(pos)
pdf1 = rvx21.pdf(pos)
# 混合比率 pi=[0.6, 0.4] に基づく全体の確率密度
pdf_total = pi[0] * pdf0 + pi[1] * pdf1

# 等高線のプロット
# クラス0 (赤系), クラス1 (青系), 全体 (緑系)
contour0 = plt.contour(x1_grid, x2_grid, pdf0, levels=8, colors='red', alpha=0.6, linestyles='dashed')
contour1 = plt.contour(x1_grid, x2_grid, pdf1, levels=8, colors='blue', alpha=0.6, linestyles='dashed')
contour_total = plt.contour(x1_grid, x2_grid, pdf_total, levels=10, colors='green', alpha=0.8, linewidths=2)

plt.clabel(contour_total, inline=True, fontsize=8, fmt='%.3f')

In [ ]:
plt.figure(figsize=(10, 8))

# 1. 2次元グリッドに対して確率密度関数を2次元の形状で再計算
pos = np.dstack((x1_grid, x2_grid))
pdf0_2d = rvx20.pdf(pos).reshape(x1_grid.shape)
pdf1_2d = rvx21.pdf(pos).reshape(x1_grid.shape)
pdf_total_2d = (pi[0] * pdf0_2d) + (pi[1] * pdf1_2d)

# 2. 実際のデータ（X2, y2）を散布図としてプロット (クラスごとに赤と青で色分け)
plt.scatter(X2[y2==0, 0], X2[y2==0, 1], alpha=0.5, color='red', edgecolor='black', label='Sample Class 0 (y=0)')
plt.scatter(X2[y2==1, 0], X2[y2==1, 1], alpha=0.5, color='blue', edgecolor='black', label='Sample Class 1 (y=1)')

# 3. 真の確率密度関数の等高線（コントゥア）を重ねて描画
contour0 = plt.contour(x1_grid, x2_grid, pdf0_2d, levels=8, colors='red', alpha=0.5, linestyles='dashed')
contour1 = plt.contour(x1_grid, x2_grid, pdf1_2d, levels=8, colors='blue', alpha=0.5, linestyles='dashed')
contour_total = plt.contour(x1_grid, x2_grid, pdf_total_2d, levels=10, colors='green', alpha=0.8, linewidths=2)

# 等高線のラベルを追加
plt.clabel(contour_total, inline=True, fontsize=8, fmt='%.3f')

# タイトル・凡例・グリッド等の設定
plt.title('2次元混合ガウス分布：等高線とサンプルの重ね合わせ', fontsize=14)
plt.xlabel('x1')
plt.ylabel('x2')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.axis('equal')
plt.savefig("gmm2d-discr.pdf", format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

x = np.linspace(X2[:, 0].min() - 1, X2[:, 0].max() + 1, 150)
y = np.linspace(X2[:, 1].min() - 1, X2[:, 1].max() + 1, 150)
X_grid, Y_grid = np.meshgrid(x, y)

pos = np.dstack((X_grid, Y_grid))

Z = gmm2.weights_[0] * rvx20.pdf(pos) + gmm2.weights_[1] * rvx21.pdf(pos)

# 3Dプロット用の静的なグラフを作成
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# 3D曲面を描画 (カラーマップはViridis)
surf = ax.plot_surface(X_grid, Y_grid, Z, cmap='viridis', edgecolor='none', alpha=1)

# カラーバーの追加
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, label='確率密度')

# タイトルと軸ラベルの設定
ax.set_title('2次元混合ガウス分布の確率密度関数', fontsize=14)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('確率密度')

# 鳥瞰図の視点角度を設定 (初期設定として仰角30度、方位角-45度などに見やすい角度をセット)
ax.view_init(elev=30, azim=-45)

plt.show()

In [ ]:
x = np.linspace(X2[:, 0].min() - 1, X2[:, 0].max() + 1, 150)
y = np.linspace(X2[:, 1].min() - 1, X2[:, 1].max() + 1, 150)
X_grid, Y_grid = np.meshgrid(x, y)
pos = np.dstack((X_grid, Y_grid))

Z = gmm2.weights_[0] * rvx20.pdf(pos) + gmm2.weights_[1] * rvx21.pdf(pos)

# Plotlyを用いてインタラクティブな3D曲面プロットを作成
fig_3d = go.Figure(data=[go.Surface(x=X_grid[0, :], y=Y_grid[:, 0], z=Z, colorscale='Viridis')])

# レイアウトの設定（タイトルや軸ラベル）
fig_3d.update_layout(
    title="確率密度関数",
    scene=dict(
        xaxis_title='x1',
        yaxis_title='x2',
        zaxis_title='Density'
    ),
    width=900,
    height=700,
    margin=dict(l=65, r=50, b=65, t=90)
)

# グラフを表示
fig_3d.show()